In [28]:
"""
测陷阱库FDR
"""
from utils import *
import os

In [216]:
"""
pFind
"""
# ori
# pfind_res_path = "D:/pFindWorkspace/normal/Gygi_2015_NBT/restrict-ori/result/open/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/Astral/ori/top50-complete/result/restricted/pFind-Filtered.spectra"
# pfind_res_path = "G:/pFindWorkspace/SingleCell/PXD039066/new/result/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/normal/Gygi_2015_NBT/restrict-new/result/restricted-0628/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/TIMSTOF/Mann2021/ori/result/open/pFind-Filtered.spectra"

# new
# pfind_res_path = "D:/pFindWorkspace/normal/Gygi_2015_NBT/restrict-new/result/open-0628/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/normal/Gygi_2015_NBT/restrict-new/result/restricted-0628/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/HLA/C3N02671-new/result/restricted-0628/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/MetaProteomics/humangut-new/result/restricted/pFind-Filtered.spectra"
pfind_res_path = "G:/pFindWorkspace/SingleCell/PXD023366/new/result/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/Astral/new/top50-complete/result/open-0628/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/plasma/PXD053748/new/result/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/normal/human_exo/human_normal-new/result/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/Astral/new/top50-complete/result/restricted-0627/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/Astral/new/top75/result/restricted/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/Astral/new/top100/result/restricted/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/Astral/new/cycle/result/restricted/pFind-Filtered.spectra"
# pfind_res_path = "D:/pFindWorkspace/plasma/PXD053748/new/result/restrict/pFind-Filtered.spectra"

trapped_num = 0
peptides = set()
seqs = set()

f = open(pfind_res_path,"r")
next(f,None)
for i,line in enumerate(f):
    items = line.split("\t")
    peptide = items[5]+items[10]  # seq+mod
    proteins = items[12].split("/")[:-1]
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break

    if peptide in peptides:
        continue
    elif flag:
        trapped_num += 1
    peptides.add(peptide)
    seqs.add(items[5])
f.close()

print("鉴定到的target肽段数目:", len(peptides))
print("其中陷阱库肽段数目:", trapped_num)
print("鉴定到的target序列数目:", len(seqs))

鉴定到的target肽段数目: 27008
其中陷阱库肽段数目: 142
鉴定到的target序列数目: 25869


In [169]:
#!/usr/bin/env python3
"""
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! 真实的肽段+序列层次！！！！！！！！！！！！！！！！！from interact-[filename].pep.xml
"""
import sys
import re
import os

def extract(xml_paths):
    total_result = list()
    for xml_path in xml_paths:
        print(xml_path)
        f = open(xml_path,"r")
        xml = f.read()
        # 把整块 <search_hit ...> ... </search_hit> 切出来
        hit_blocks = re.findall(r'<search_hit(.*?)>(.*?)</search_hit>', xml, flags=re.S)
        for attr, inner in hit_blocks:
            # --- 1. peptide ---
            seq = re.search(r'peptide="([^"]*)"', attr).group(1)
            res = re.search(r'<modification_info[^>]*modified_peptide="([^"]*)"', inner)
            if res is None:
                pep = seq
            else:
                pep = res.group(1)
    
            # --- 2. 所有蛋白 ---
            proteins = []
            # 主蛋白
            proteins.append(re.search(r'protein="([^"]*)"', attr).group(1))
            # alternative 蛋白
            proteins += re.findall(r'<alternative_protein[^>]*\sprotein="([^"]*)"', inner)
            
            # --- 3. probability ---
            prob = re.search(r'<peptideprophet_result[^>]*\sprobability="([^"]*)"', inner)
            prob = prob.group(1) if prob else ''
    
            total_result.append((pep, seq, proteins, float(prob)))
        f.close()
    return total_result


# msfragger_res_dir = "G:/MSFraggerWorkspace/Astral/top50-restricted"
# msfragger_res_dir = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/restricted"
# msfragger_res_dir = "G:/MSFraggerWorkspace/HLA/C3N02671/restricted"
# msfragger_res_dir = "G:/MSFraggerWorkspace/MetaProteomics/humangut/restricted"
msfragger_res_dir = "G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted"

xml_paths = [os.path.join(msfragger_res_dir,filename) for filename in os.listdir(msfragger_res_dir) if filename.endswith(".pep.xml")]
total_result = extract(xml_paths)

total_result = sorted(total_result, key=lambda item: item[3], reverse=True)
fdrs = list()
t = 1e-16
d = 0
peptides = set()
for result in total_result:
    peptide = result[0]
    if peptide in peptides:
        fdrs.append(d/t)
        continue
    peptides.add(peptide)
    proteins = result[2]
    flag = True
    for protein in proteins:
        if not protein.startswith("rev_"):
            flag = False
            break
    if flag:
        d += 1
    else:
        t += 1
    fdrs.append(d/t)



min_fdr = 100
for i in range(len(fdrs)-1,-1,-1):
    fdrs[i] = min(fdrs[i], min_fdr)
    if fdrs[i]<=0.01:
        lim = i
        break
# for i, fdr in enumerate(fdrs):
#     if fdr>0.01:
#         lim = i
#         break
print(t,d,lim)

decoy_peptides = set()
peptides = set()
seqs = set()
decoy_num = 0
trapped_num = 0
for i,result in enumerate(total_result):
    if i>lim:
        break
    peptide = result[0]
    if peptide in peptides:
        continue
    
    proteins = result[2]
    flag = True
    for protein in proteins:
        if not protein.startswith("rev_"):
            flag = False
            break
    if flag:
        if not peptide in decoy_peptides:
            decoy_peptides.add(peptide)
            decoy_num += 1
        continue

    peptides.add(peptide)
    seqs.add(result[1])
    # flag = False   # target peptide
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = True
    if not flag:
        trapped_num += 1
    

print("鉴定到的target肽段数目:", len(peptides))
print("鉴定到的target肽段序列数目:", len(seqs))
print("其中陷阱库肽段数目:", trapped_num)
print("decoy肽段数目", decoy_num)

G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4386-1.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4386-2.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4386-3.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4388-1.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4388-2.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4909-1.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4909-2.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-4909-3.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-5689.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-5898-1.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-5898-2.pep.xml
G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted\interact-GV-9432.pep.xml
28330.0 3037 189704
鉴定到的target肽段

In [145]:
"""
MSFragger(psm.tsv): 
            psm.tsv卡PSM层次0.01FDR？
            peptides.tsv中都是unique peptide sequence，卡序列层次0.01FDR？
            ion.tsv中不是unique peptide，卡母离子(电荷不一致也算不一样)层次FDR？
"""
# msfragger_res_path = "D:/MSFraggerWorkspace/TIMSTOF/Mann2021/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/Astral/top50-restricted/psm.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/open/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/restricted/psm.tsv"
msfragger_res_path = "G:/MSFraggerWorkspace/HLA/C3N02671/restricted-dl/psm.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/MetaProteomics/humangut/restricted-dl/ion.tsv"
seqs = set()
peptides = set()
num = 0
trapped_num = 0
decoy_num = 0

f = open(msfragger_res_path, "r")
next(f,None)
for line in f:
    items = line.split("\t")
    seq = items[2]
    peptide = items[2]+"\t"+items[26]
    
    if peptide in peptides:
        continue
    peptides.add(peptide)
    seqs.add(seq)

    # 获取蛋白质列表
    proteins = [items[30]]
    for protein in items[-1].split(","):
        if protein != '\n':
            proteins.append(protein.strip())
    
    flag = False
    for protein in proteins:
        if not protein.startswith("REV_"):
            flag = True
            break
    if not flag:
        decoy_num += 1
        continue

    num += 1
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag:
        # print(peptide, proteins)
        trapped_num += 1
        
f.close()

print("鉴定到的target肽段数目:", len(peptides))
print("鉴定到的target肽段序列数目:", len(seqs))
print("其中陷阱库肽段数目:", trapped_num)
print("decoy肽段数目", decoy_num)

鉴定到的target肽段数目: 21863
鉴定到的target肽段序列数目: 21860
其中陷阱库肽段数目: 0
decoy肽段数目 0


In [171]:
"""
MSFragger(ion.tsv): peptides.tsv中都是unique peptide sequence，卡序列层次0.01FDR？
            ion.tsv中不是unique peptide，卡母离子(电荷不一致也算不一样)层次FDR？
"""
# msfragger_res_path = "D:/MSFraggerWorkspace/TIMSTOF/Mann2021/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/Astral/top50-restricted/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/open/ion.tsv"
msfragger_res_path = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/restricted/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/HLA/C3N02671/restricted-dl/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/SingleCell/PXD023366/restricted-dl/ion.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/MetaProteomics/humangut/restricted/ion.tsv"
seqs = set()
peptides = set()
num = 0
trapped_num = 0
decoy_num = 0

f = open(msfragger_res_path, "r")
next(f,None)
for line in f:
    items = line.split("\t")
    peptide = items[0]+"\t"+items[14]
    seq = items[0]
    
    if peptide in peptides:
        continue
    peptides.add(peptide)
    seqs.add(seq)

    # 获取蛋白质列表
    proteins = [items[16]]
    for protein in items[-1].split(","):
        if protein != '\n':
            proteins.append(protein.strip())
    
    flag = False
    for protein in proteins:
        if not protein.startswith("REV_"):
            flag = True
            break
    if not flag:
        decoy_num += 1
        continue

    num += 1
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag:
        # print(peptide, proteins)
        trapped_num += 1
        
f.close()

print("鉴定到的target肽段数目:", len(peptides))
print("鉴定到的target肽段序列数目:", len(seqs))
print("其中陷阱库肽段数目:", trapped_num)
print("decoy肽段数目", decoy_num)

鉴定到的target肽段数目: 145353
鉴定到的target肽段序列数目: 134214
其中陷阱库肽段数目: 878
decoy肽段数目 0


In [139]:
"""
MSFragger(peptide.tsv): 
            peptides.tsv中都是unique peptide sequence，卡序列层次0.01FDR？
            ion.tsv中不是unique peptide，卡母离子(电荷不一致也算不一样)层次FDR？
            前者似乎更严格一些，所以鉴定到的肽段数目会少于ion.tsv
"""
# msfragger_res_path = "D:/MSFraggerWorkspace/TIMSTOF/Mann2021/peptide.tsv"
msfragger_res_path = "G:/MSFraggerWorkspace/Astral/top50-restricted/peptide.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/open/peptide.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/normal/Gygi_2015_NBT/restricted/peptide.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/HLA/C3N02671/restricted/peptide.tsv"
# msfragger_res_path = "G:/MSFraggerWorkspace/MetaProteomics/humangut/restricted/peptide.tsv"
num = 0
trapped_num = 0
decoy_num = 0

f = open(msfragger_res_path, "r")
next(f,None)
for line in f:
    items = line.split("\t")
    peptide = items[0]+"\t"+items[10]  # seq+mod
    
    proteins = [items[12]]
    for protein in items[-1].split(","):
        if protein != '\n':
            proteins.append(protein.strip())

    flag = False
    for protein in proteins:
        if not protein.startswith("rev_"):
            flag = True
            break
    if not flag:
        decoy_num += 1
        continue

    num += 1
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag:
        # print(peptide, proteins)
        trapped_num += 1
f.close()

print("鉴定到的target肽段数目:", num)
print("其中陷阱库肽段数目:", trapped_num)
print("decoy肽段数目", decoy_num)

鉴定到的target肽段数目: 34050
其中陷阱库肽段数目: 150
decoy肽段数目 0


In [20]:
"""
MaxQuant: evidence.txt
"""
maxquant_path = "D:/data/ms/Astral/PXD046357-Single_Cell/top50/combined/txt/evidence.txt"
f = open(maxquant_path, "r")
next(f,None)

peptides = set()
decoy_num = 0
trapped_num = 0
for line in f:
    items = line.strip().split("\t")
    peptide = items[9]
    proteins = items[15].split(";")

    flag = False
    for protein in proteins:
        if protein == "":
            flag = True
            break
    if flag:
        decoy_num += 1
        continue
    
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag and peptide not in peptides:
        trapped_num += 1
    peptides.add(peptide)
f.close()

print("鉴定到的target肽段数目:", len(peptides))
print("其中陷阱库肽段数目:", trapped_num)
print("decoy肽段数目", decoy_num)

鉴定到的target肽段数目: 20814
其中陷阱库肽段数目: 59
decoy肽段数目 59


In [1]:
"""
MaxQuant: msms.txt
"""
# maxquant_path = "G:/MaxQuantWorkspace/Astral/top50/combined/txt/msms.txt"
# maxquant_path = "G:/MaxQuantWorkspace/normal/Gygi_2015_NBT/msms.txt"
# maxquant_path = "G:/MaxQuantWorkspace/HLA/C3N02671/msms.txt"
maxquant_path = "G:/MaxQuantWorkspace/SingleCell/PXD023366/combined/txt/msms.txt"
f = open(maxquant_path, "r")
next(f,None)

peptides = set()
seqs = set()
decoy_num = 0
trapped_num = 0
num = 0
for line in f:
    num += 1
    items = line.strip().split("\t")
    peptide = items[7]
    proteins = items[12].split(";")

    if len(proteins)==0 or proteins[0] == "":
        decoy_num += 1
        continue
    
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag and peptide not in peptides:
        trapped_num += 1
    peptides.add(peptide)

    # 提取肽段序列
    seq = ""
    count = 0
    for i in peptide:
        if i == '(':
            count += 1
            continue
        elif i == ')':
            count -= 1
            continue
        if count == 0:
            seq += i
    seqs.add(seq)
f.close()

print("鉴定到的target肽段数目:", len(peptides))
print("鉴定到的target肽段序列数目:", len(seqs))
print("其中陷阱库肽段数目:", trapped_num)
print("decoy肽段数目", decoy_num)

鉴定到的target肽段数目: 15923
鉴定到的target肽段序列数目: 15657
其中陷阱库肽段数目: 40
decoy肽段数目 137


In [66]:
"""
Comet + Percolator：
    -r 指定的输出是peptide unique的; 其只包含target肽段
"""
# result_path = "G:/CometWorkspace/Astral/top50/peptide_results.txt"  
# result_path = "G:/CometWorkspace/SingleCell/PXD023366/peptide_results.txt"
# result_path = "G:/CometWorkspace/Gygi_2015_NBT/peptide_results.txt"
# result_path = "G:/CometWorkspace/humangut/peptide_results.txt"
# result_path = "D:/data/ms/MetaProteomics/humangut/1/peptide_results.txt"
result_path = "G:/CometWorkspace/HLA/c3n02671/peptide_results.txt"
result_dir = os.path.dirname(result_path)
for file in os.listdir(result_dir):
    if file.endswith(".pin"):
        print(result_dir+'/'+file, end=" ")
print()
peptides = set()
seqs = set()
decoy_num = 0
trapped_num = 0

f = open(result_path, "r")
next(f,None)
for line in f:
    items = line.strip().split("\t")
    q_value = float(items[2])
    if q_value > 0.01:
        break
    pep = items[4][2:-2]
    
    proteins = items[5:]
    flag = False
    for protein in proteins:
        if not protein.startswith("DECOY_"):
            flag = True
            break
    if not flag:
        decoy_num += 1
        continue
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag and pep not in peptides:
        trapped_num += 1
    peptides.add(pep)

    # 提取序列
    seq = ""
    i = 0
    while i<len(pep):
        if pep[i] == '[':
            i = pep.find(']',i+1) + 1
            continue
        seq += pep[i]
        i += 1
    seqs.add(seq)
f.close()

print("鉴定到的target肽段数目:", len(peptides))
print("其中陷阱库肽段数目:", trapped_num)
print("鉴定到的target肽段序列数目:", len(seqs))

G:/CometWorkspace/HLA/c3n02671/20180425_QE_HFX_LC2_HLAIIp_CHC_SA_normC3N02671_10_R1.pin G:/CometWorkspace/HLA/c3n02671/20180425_QE_HFX_LC2_HLAIIp_CHC_SA_normC3N02671_10_R2.pin G:/CometWorkspace/HLA/c3n02671/20180425_QE_HFX_LC2_HLAIIp_CHC_SA_normC3N02671_10_R3.pin G:/CometWorkspace/HLA/c3n02671/20180425_QE_HFX_LC2_HLAIp_CHC_SA_normC3N02671_10_R1.pin G:/CometWorkspace/HLA/c3n02671/20180425_QE_HFX_LC2_HLAIp_CHC_SA_normC3N02671_10_R2.pin G:/CometWorkspace/HLA/c3n02671/20180425_QE_HFX_LC2_HLAIp_CHC_SA_normC3N02671_10_R3.pin G:/CometWorkspace/HLA/c3n02671/20180426_QE_HFX_LC2_HLAIIp_CHC_SA_tumC3N02671_08_R1.pin G:/CometWorkspace/HLA/c3n02671/20180426_QE_HFX_LC2_HLAIIp_CHC_SA_tumC3N02671_08_R2.pin G:/CometWorkspace/HLA/c3n02671/20180426_QE_HFX_LC2_HLAIIp_CHC_SA_tumC3N02671_08_R3.pin G:/CometWorkspace/HLA/c3n02671/20180426_QE_HFX_LC2_HLAIp_CHC_SA_tumC3N02671_08_R1.pin G:/CometWorkspace/HLA/c3n02671/20180426_QE_HFX_LC2_HLAIp_CHC_SA_tumC3N02671_08_R2.pin G:/CometWorkspace/HLA/c3n02671/20180426_QE

In [52]:
pep = "abc[]asdfcas[adfas]dasc[sd]a"
seq = ""
i = 0
while i<len(pep):
    if pep[i] == '[':
        i = pep.find(']',i+1) + 1
        print(i)
        continue
    seq += pep[i]
    i += 1
# seqs.add(seq)
print(seq)

5
19
27
abcasdfcasdasca


In [22]:
for i,pep in enumerate(peptides):
    print(pep)
    if i>100:
        break

DICGDCDLIIEAALENMEIKK
IAETAITTGNIGYAVNSFTQK
EMADLEQALLDGTSIREDEVLAK
HEIGQAAVNSLR
WVEDGKINIDDNIMK
M[15.9949]ALTN[0.9840]AQYDSLMR
ELMDAVDDYIPTPDR
AQM[15.9949]GMILGN[0.9840]NSTGK
MVEPERVIEFR
IIATHVGM[15.9949]TPEVGQQNTEGSLEVNLLPQGTLAECVR
FGADTDNWM[15.9949]WPR
MDPYVDEFQDHGGSLIM[15.9949]LAK
LAEIQDNSIGR
TVGSDIAN[0.9840]GEIMVMSFDTTN[0.9840]AGLTDTLAGK
VVWIDGSKEQLDALTEEVTSLPEGNR
AGGVKPTPNN[0.9840]GAVPTAGCYLQYTATDSGK
LAVYHEQTQPLIEYYDK
MMNGIGGSGDFTR
GIPQIYYGTELLMN[0.9840]GTK
EDELIVGLQTDEPLKR
NIDKNDSLEFR
FEDIASFEDADMLMIPGGM[15.9949]PGSTNLNEHEGVR
DAGLDASQINEVILVGGSTR
ADGLAN[0.9840]FHNAFTGIYEAR
VFVLDSLSTGPEQR
YGITGATEIIHNPSYELLFEEETK
GLAADDVVASDDGK
ITFTYLGMTPQTVK
GM[15.9949]GKDLYENSALAK
AQLIDKDGVDVK
AEGSYILPDNVPSNEFLN[0.9840]LEGDKISTSR
GQN[0.9840]DDWASVAASNGTSTK
THNDGVFDAYTPEM[15.9949]K
AAGVSDIGAALTGNLPGVVTVQGN[0.9840]GM[15.9949]PGEEEPQIIIR
EIDWANLSFGYM[15.9949]K
DGVLSETAR
NM[15.9949]PEFSEELASLADAYVDDAFGSCHR
YTWVGPLN[0.9840]YSDGSAQVQVESEGR
AYNHETGAVYNPSK
THLAN[0.9840]LPIYYEYK
GTAVMVPEWDAEK
AECVIINAVE

In [69]:
scores = [10,9,8,7,6,5,4,3,2,1]
scores = [10,5,5,4,4,3,3,2,2,1]
scores = [10]

def rescore1(scores):
    rescores = list()
    n = len(scores)
    summ = 0
    for score in scores:
        summ += score

    summ /= n
    for score in scores:
        rescores.append(score/summ)
    return rescores


def rescore2(scores):
    rescores = list()
    n = len(scores)
    summ = 0
    for score in scores:
        summ += score

    summ /= n
    for score in scores:
        rescores.append(score-summ)
    return rescores


def rescore3(scores):
    rescores = list()
    n = len(scores)
    summ = 0
    for score in scores:
        summ += score

    for score in scores:
        rescores.append(n*score/summ-1)
    return rescores

print(rescore1(scores))
print(rescore2(scores))
print(rescore3(scores))

[1.0]
[0.0]
[0.0]


In [63]:
# f = open("D:/pFindWorkspace/normal/human_exo/human_normal-new/result/human-normal-new.F1.L1.qry.res","r")
f = open("D:/pFindWorkspace/Astral/new/top50-complete/result/top50-complete.F1.L1.qry.res", "r")
seq_decoys = None
total = 0
num = 0
while True:
    line = f.readline()
    if not line:
        break
    if line.startswith("S"):
        total += 1
        if seq_decoys is not None:
            seq_decoys = sorted(seq_decoys, key=lambda item:item[0])
            for i in range(1,len(seq_decoys)):
                if seq_decoys[i][0] == seq_decoys[i-1][0] and decoy > 0:
                    num += 1
                    # print(spectrum_name,seq)
                    break
        spectrum_name = f.readline().strip()
        seq_decoys = list()
    else:
        items = line.strip().split("\t")
        mod_num = int(items[6])
        decoy = int(items[8+mod_num*2])
        seq = items[1]
        seq_decoys.append((seq,decoy))
        
f.close()
print(total, num)

400483 29163


In [99]:
fr = open("D:/data/fasta/humangut/Human_gut.fasta_td.fasta","r")
fw = open("D:/data/fasta/humangut/Human_gut.fasta","w")
while True:
    line = fr.readline()
    if not line:
        break
    if line.startswith(">REV_"):
        line = fr.readline()
    else:
        fw.write(line)
fr.close()
fw.close()

In [107]:
fr = open("D:/data/fasta/humangut/Human_gut.fasta","r")
stage = 60000000
num = 0
pro_num = 5192079
pro_id = 1
while True:
    line = fr.readline()
    if not line:
        break
    if not line.startswith(">"):
        num += len(line.strip())
        if num > stage:
            stage += 60000000
            print(pro_id/pro_num)
    else:
        pro_id += 1
fr.close()
print(num, pro_num)

0.005326960548943882
0.014182950606105955
0.0249391428751373
0.03731299157813277
0.05119182508586637
0.06641848092064855
0.0829775509964313
0.10081125499053462
0.11988723592225774
0.14015426190549105
0.16166529823602452
0.1844030108170542
0.2082890110108109
0.23327553375054577
0.2593136583630565
0.28644787569680663
0.31472036538735254
0.34415019494117866
0.374817293804659
0.4068091028661159
0.4402117148063425
0.4750411155146137
0.5114144064448942
0.5493550849284073
0.5889640739287673
0.6304179886322994
0.6739783812996682
0.7197635089912923
0.7679858106935584
0.8188721704735232
0.8727255498231056
0.9298358133610833
0.9904945591159149
1989077052 5192079


In [203]:
# f = open("D:/pFindWorkspace/Astral/new/top50-complete/result/top50-complete.F2.L1.sub.fasta","r")
f = open("D:/pFindWorkspace/Astral/ori/top50-complete/result/top50-complete.F3.L1.sub.fasta","r")
human_num = 0
arath_num = 0
for line in f:
    if line.startswith(">"):
        if line.split("\t")[0].endswith("_HUMAN"):
            human_num += 1
        elif line.split("\t")[0].endswith("_ARATH"):
            arath_num += 1
f.close()
print(human_num, arath_num, arath_num/(human_num+arath_num))

5532 1903 0.2559515803631473


In [237]:
spectrum2res = dict()
fo = open("D:/pFindWorkspace/Astral/new/top50-complete/result/top50-complete.F1.L1.ot","r")
while True:
    line = fo.readline()
    if not line:
        break
    if line.startswith("S"):
        spectrum_name = fo.readline().strip()
        flag = False
    elif not flag:
        items = line.strip().split("\t")
        mod_num = int(items[6])
        mods = ""
        for i in range(mod_num):
            mods += f"{items[7+2*i]},{items[8+2*i]};"
        seq = items[1]
        modified_seq = seq+"\t"+mods
        fine_score = float(items[3])
        spectrum2res[spectrum_name] = [modified_seq,fine_score]
        flag = True
fo.close()

fs = open("D:/pFindWorkspace/Astral/new/top50-complete/result/top50-complete.F1.L1.sa","r")
while True:
    line = fs.readline()
    if not line:
        break
    if line.startswith("S"):
        spectrum_name = fs.readline().strip()
        flag = False
    elif not flag:
        fine_score = float(items[3])
        items = line.strip().split("\t")
        mod_num = int(items[6])
        mods = ""
        for i in range(mod_num):
            mods += f"{items[7+2*i]},{items[8+2*i]};"
        seq = items[1]
        modified_seq = seq+"\t"+mods
        if spectrum_name not in spectrum2res:
            spectrum2res[spectrum_name] = [modified_seq,fine_score]
        elif fine_score>spectrum2res[spectrum_name][1]:
            spectrum2res[spectrum_name] = [modified_seq,fine_score]
        flag = True
fs.close()      

fq = open("D:/pFindWorkspace/Astral/new/top50-complete/result/top50-complete.F1.L1.qry.res","r")
while True:
    line = fq.readline()
    if not line:
        break
    if line.startswith("S"):
        spectrum_name = fq.readline().strip()
        target_pep = spectrum2res[spectrum_name][0] if spectrum_name in spectrum2res else None
    elif target_pep is not None:
        items = line.strip().split("\t")
        mod_num = int(items[6])
        mods = ""
        for i in range(mod_num):
            mods += f"{items[7+2*i]},{items[8+2*i]};"
        seq = items[1]
        modified_seq = seq+"\t"+mods

        if modified_seq == target_pep:
            is_decoy = (items[8+mod_num*2]!='0')
            spectrum2res[spectrum_name].append(is_decoy)
fq.close()

fu = open("D:/pFindWorkspace/Astral/new/top50-complete/result/pFind.spectra","r")
next(fu,None)
for line in fu:
    items = line.strip().split("\t")
    spectrum_name = items[0]
    if len(items) <= 5:
        spectrum2res.pop(spectrum_name,"")
        continue
    if spectrum_name in spectrum2res:
        proteins = items[12].strip("/").split("/")
        spectrum2res[spectrum_name].append(proteins)
fu.close()

spectrum_res = sorted(spectrum2res.items(), key=lambda items: items[1][1], reverse=True)
q_values = list()
t = 1e-16
d = 0
for spectrum, (modified_seq,fine_score,is_decoy,proteins) in spectrum_res:
    if is_decoy:
        d += 1
    else:
        t += 1
    q_values.append(d/t)
min_q_value = 1e6
for i in range(len(q_values)-1,-1,-1):
    q_values[i] = min_q_value = min(q_values[i],min_q_value)

peptides = set()
for (spectrum_name,(peptide,fine_score,is_decoy,proteins)),q_value in zip(spectrum_res,q_values):
    flag = False
    for protein in proteins:
        if not protein.startswith("REV_"):
            flag = True
            break
    if not flag:
        continue
    
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break
    if flag:
        print(peptide, proteins)
        trapped_num += 1
        if peptide in peptides:
            continue
        else:
            peptides.add(peptide)

ValueError: too many values to unpack (expected 3)

In [277]:
f = open("D:/data/fasta/pure/uniprotkb_human_20435.fasta", "r")
total_num = 0
plasma_num = 0
for line in f:
    if line.startswith(">"):
        total_num += 1
        if "plasma" in line.strip().lower():
            plasma_num += 1
print(total_num, plasma_num)
f.close()

20435 15


In [22]:
# proteins = set()
f = open("D:/data/ms/plasma/PXD059640/library.tsv","r")
next(f,None)
max_len = 0
min_len = 100
max_miss = 0
for line in f:
    items = line.strip().split("\t")
    min_len = min(min_len,len(items[5]))
    max_len = max(max_len,len(items[5]))
    miss = 0
    for i in items[5]:
        if i=='K' or i=='R':
            miss += 1
    max_miss = max(max_miss,miss)
    # proteins.add(items[3])
f.close()
# print(len(proteins))
print(min_len,max_len, max_miss)

7 50 3


In [285]:
mods = set()
f = open("D:/data/ms/Hela-3T3/library.tsv/library.tsv","r")
next(f,None)
for line in f:
    items = line.strip().split("\t")
    modified_seq = items[6]
    if not '(' in modified_seq:
        continue
    mod = modified_seq.split("(")[1].split(")")[0]
    mods.add(mod)
f.close()
print(mods)

{'UniMod:4', 'UniMod:1', 'UniMod:35'}


In [289]:
zuzhi = set()
f = open("D:/data/ms/Hela-3T3/samples.sdrf","r")
next(f,None)
for line in f:
    zuzhi.add(line.strip().split("\t")[2])
f.close()
print(zuzhi, len(zuzhi))

{'Lymphocytes', 'Muscle', 'Skin', 'Blood plasma', 'Kidney', 'Erythrocytes', 'Colon', 'Neutrophils', 'Macrophages', 'Urinary.Bladder', 'Spleen', 'Stomach', 'Depl.Plasma', 'Platelets', 'Prostate', 'Bone.Marrow', 'Brain', 'T-cell CD8', 'Adipose', 'Liver', 'Monocytes', 'Macrophage', 'Nerve', 'Ovary', 'T-cell CD4', 'Neutrophil', 'Erythrocyte', 'Lung', 'Pancreas', 'Bone marrow', 'Heart', 'Artery', 'Monocyte', 'Urinary bladder', 'Adipose tissue', 'Aorta', 'B-cell'} 37


In [299]:
f1 = open("D:/pFindWorkspace/normal/human_exo/human_normal-new/result/human-normal-new.F1.L1.sa","r")
f2 = open("D:/pFindWorkspace/normal/human_exo/human_normal-new/result/human-normal-new.F1.L10.sa","r")
for i,(line1, line2) in enumerate(zip(f1,f2)):
    items1, items2 = line1.split("\t"), line2.split("\t")
    if items1[0]!=items2[0]:
        print(i)
        break
f1.close()
f2.close()

6708


In [2]:
peptides = set()

fr = open("D:/pFindWorkspace/HLA/C3N02671-new/result/restricted-0628/pFind-Filtered.spectra", "r")
for line in fr:
    peptide = line.strip().split("\t")[5]
    peptides.add(peptide)
fr.close()

fw = open("D:/pFindWorkspace/HLA/C3N02671-new/result/restricted-0628/peptide.txt", "w")
for idx,peptide in enumerate(peptides):
    fw.write(f">peptide_{idx}\n")
    fw.write(peptide+"\n")
fw.close()

In [6]:
f = open("D:/pFindWorkspace/MetaProteomics/humangut-new/result/humangut-new.F5.L1.sa","r")
# f = open("D:/pFindWorkspace/HLA/C3N02671-new/result/restricted-0628/C3N02671-new.F1.L1.sa","r")
# f = open("D:/pFindWorkspace/HLA/C3N02671-ori/result/restricted/C3N02671-ori.F1.L1.sa","r")
# f = open("D:/pFindWorkspace/Astral/new/top50-complete/result/restricted-0627/top50-complete.F1.L1.sa","r")  # 0
peptides = set()
total = 0
hit = 0
k = 0
for line in f:
    if line.startswith("S"):
        flag = False
        total += 1
        peptides.clear()
        continue
    items = line.strip().split("\t")
    if len(items)==1:
        continue
    seq = items[1]
    mod_num = int(items[6])
    mod = ""
    for i in range(7,7+mod_num*2,2):
        mod += (items[i]+","+items[i+1])
    peptide = seq+"\t"+mod
    # print(peptide)
    # break
    if peptide in peptides:
        k += 1
        if not flag:
            flag = True
            hit += 1
            print(peptide)
            break
    else:
        peptides.add(peptide)
f.close()
print(total, k, hit)

124656 0 0


D:/pFindWorkspace/normal/Gygi_2015_NBT/restrict-new/result/restricted-0628/restrict-new.F1.L1.qry.res
D:/pFindWorkspace/normal/Gygi_2015_NBT/restrict-new/result/restricted-0628/restrict-new.F10.L1.qry.res
2487517 1913443 574074


In [244]:
num / total

0.34507783410470827

In [248]:
num / total

0.547773556696838

In [254]:
num / total

0.19092076872056218

In [258]:
num / total

0.5637658148221721

In [262]:
num / total

0.23078194038472902

In [276]:
import os

dirname = "G:/data/ms/9species/9Bacillus/"
names = set()
for filename in os.listdir(dirname):
    if filename.endswith(".pf2"):
        names.add(filename.rsplit("_HCDFT.pf2", 1)[0])
# print(names)
for filename in os.listdir(dirname):
    if filename.endswith(".raw"):
        name = filename.strip(".raw")
        if name not in names:
            print(name)

150723_QE1_PK_Bsub_Exp_PF_6
150723_QE1_PK_Bsub_Heat_1_DDA
150723_QE1_PK_Bsub_Heat_2_DDA
150723_QE1_PK_Bsub_Heat_PF_1
150723_QE1_PK_Bsub_Heat_PF_2
150723_QE1_PK_Bsub_Heat_PF_3
150723_QE1_PK_Bsub_Heat_PF_4
150723_QE1_PK_Bsub_Heat_PF_5
150723_QE1_PK_Bsub_Spore_1_DDA


In [49]:
minn = 100
maxn = -1
f = open("G:/MSFraggerWorkspace/Astral/top50-restricted/psm.tsv","r")
# f = open("G:/MSFraggerWorkspace/Astral/top50-restricted/peptide.tsv","r")
# f = open("G:/MSFraggerWorkspace/Astral/top50-restricted/ion.tsv","r")
next(f,None)
for line in f:
    items = line.strip().split("\t")
    probability = float(items[20])   # psm.tsv
    # probability = float(items[7])   # peptide.tsv
    # probability = float(items[10])  # ion.tsv
    minn = min(minn, probability)
    maxn = max(maxn, probability)
f.close()
print(minn, maxn)

0.8848 1.0
